In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

In [13]:
results_path = Path("../data/phase2_results.json")

with results_path.open("r", encoding="utf-8") as f:
    results = json.load(f)

len(results), results[0].keys()

(21,
 dict_keys(['id', 'categoria_esperada', 'cumple_normas_esperado', 'texto_preview', 'raw_response', 'scores', 'predicted_top_clinical']))

In [14]:
df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 80)
df[["id", "categoria_esperada", "predicted_top_clinical", "cumple_normas_esperado", "scores"]].head()

,id,categoria_esperada,predicted_top_clinical,cumple_normas_esperado,scores
0,T01,[depresion],p_ansiedad,None,"{'p_depresion': 40, 'p_ansiedad': 60, 'p_suicida': 0, 'score_normas': 100, '..."
1,T02,[depresion],p_depresion,None,"{'p_depresion': 20, 'p_ansiedad': 10, 'p_suicida': 0, 'score_normas': 100, '..."
2,T03,"[depresion, ansiedad]",p_ansiedad,None,"{'p_depresion': 20, 'p_ansiedad': 80, 'p_suicida': 0, 'score_normas': 100, '..."
3,T04,"[suicida, depresion]",p_depresion,None,"{'p_depresion': 90, 'p_ansiedad': 20, 'p_suicida': 80, 'score_normas': 100, ..."
4,T05,[depresion],p_ansiedad,None,"{'p_depresion': 40, 'p_ansiedad': 60, 'p_suicida': 0, 'score_normas': 100, '..."


In [15]:
scores_df = pd.json_normalize(df["scores"])
df2 = pd.concat([df.drop(columns=["scores"]), scores_df], axis=1)

df2.head()

,id,categoria_esperada,cumple_normas_esperado,texto_preview,raw_response,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,razon_normas,notas
0,T01,[depresion],None,"Estaba navegando por Facebook y, en una publicación, vi la foto de mi amiga ...","{\n ""p_depresion"": 40,\n ""p_ansiedad"": 60,\n ""p_suicida"": 0,\n ""score_no...",p_ansiedad,40,60,0,100,True,NaN,"Se detectan señales de inseguridad y comparación social, lo que puede indica..."
1,T02,[depresion],None,Salí a la cocina por un vaso de agua y me enteré de que mi abuelo había sido...,"{\n ""p_depresion"": 20,\n ""p_ansiedad"": 10,\n ""p_suicida"": 0,\n ""score_no...",p_depresion,20,10,0,100,True,NaN,"La persona experimenta culpa por no sentir empatía, lo que sugiere una conci..."
2,T03,"[depresion, ansiedad]",None,Estaba haciendo un trabajo para la universidad y se me erizó la piel porque ...,"{\n ""p_depresion"": 20,\n ""p_ansiedad"": 80,\n ""p_suicida"": 0,\n ""score_no...",p_ansiedad,20,80,0,100,True,NaN,"Se observan señales de ansiedad como la angustia y el pensamiento negativo, ..."
3,T04,"[suicida, depresion]",None,"He intentado hacer de todo, cosas en las que se supone que soy bueno o suelo...","{\n ""p_depresion"": 90,\n ""p_ansiedad"": 20,\n ""p_suicida"": 80,\n ""score_n...",p_depresion,90,20,80,100,True,NaN,Se observan señales de depresión y pensamientos suicidas debido a la pérdida...
4,T05,[depresion],None,"Me he descuidado con la universidad y mi educación técnica, no supe cómo hac...","{\n ""p_depresion"": 40,\n ""p_ansiedad"": 60,\n ""p_suicida"": 0,\n ""score_no...",p_ansiedad,40,60,0,100,True,NaN,"Se observan señales de ansiedad y preocupación por el futuro académico, pero..."


In [16]:
#Función para revisar coincidencia entre categoría esperada y resultados clínicos

def clinical_match(row):
    expected = row["categoria_esperada"]

    # Casos de normas
    if "normas" in expected or "cumple_normas" in expected:
        if pd.isna(row.get("cumple_normas_esperado")):
            return np.nan
        return row["cumple_normas"] == row["cumple_normas_esperado"]

    # Casos emocionales
    labels = ["depresion", "ansiedad", "suicida"]
    present = []
    for label in labels:
        col = f"p_{label}"
        if col in row and pd.notna(row[col]):
            present.append((label, row[col]))

    if not present:
        return False

    top_label = max(present, key=lambda x: x[1])[0]
    expected_labels = expected

    # Coincide si la etiqueta dominante está dentro de las esperadas
    return top_label in expected_labels

In [17]:
df2["clinical_match"] = df2.apply(clinical_match, axis=1)
df2[["id", "categoria_esperada", "predicted_top_clinical", "clinical_match"]]

,id,categoria_esperada,predicted_top_clinical,clinical_match
0,T01,[depresion],p_ansiedad,False
1,T02,[depresion],p_depresion,True
2,T03,"[depresion, ansiedad]",p_ansiedad,True
3,T04,"[suicida, depresion]",p_depresion,True
4,T05,[depresion],p_ansiedad,False
5,T06,[depresion],p_depresion,True
6,T07,"[depresion, ansiedad]",p_depresion,True
7,T08,"[depresion, suicida]",p_suicida,True
8,T09,"[ansiedad, depresion, suicida]",p_ansiedad,True
9,T10,[depresion],p_depresion,True


In [18]:
summary_cols = [
    "id",
    "categoria_esperada",
    "predicted_top_clinical",
    "p_depresion",
    "p_ansiedad",
    "p_suicida",
    "score_normas",
    "cumple_normas",
    "cumple_normas_esperado",
    "clinical_match",
]

df2[summary_cols]

,id,categoria_esperada,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,cumple_normas_esperado,clinical_match
0,T01,[depresion],p_ansiedad,40,60,0,100,True,None,False
1,T02,[depresion],p_depresion,20,10,0,100,True,None,True
2,T03,"[depresion, ansiedad]",p_ansiedad,20,80,0,100,True,None,True
3,T04,"[suicida, depresion]",p_depresion,90,20,80,100,True,None,True
4,T05,[depresion],p_ansiedad,40,60,0,100,True,None,False
5,T06,[depresion],p_depresion,80,40,20,100,True,None,True
6,T07,"[depresion, ansiedad]",p_depresion,80,40,20,80,True,None,True
7,T08,"[depresion, suicida]",p_suicida,80,20,90,0,False,None,True
8,T09,"[ansiedad, depresion, suicida]",p_ansiedad,60,90,20,100,True,None,True
9,T10,[depresion],p_depresion,80,40,0,80,True,None,True


In [19]:
## Revisión de casos neutros
neutros = df2[df2["categoria_esperada"].apply(lambda x: "neutral" in x)]
neutros[summary_cols]



,id,categoria_esperada,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,cumple_normas_esperado,clinical_match
15,T16,[neutral],p_depresion,0,0,0,100,True,None,False
16,T17,[neutral],p_depresion,0,0,0,100,True,None,False
17,T18,[neutral],p_ansiedad,0,20,0,100,True,None,False


In [20]:
## Revisión de normas
normas = df2[df2["categoria_esperada"].apply(lambda x: "normas" in x or "cumple_normas" in x)]
normas[summary_cols]

,id,categoria_esperada,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,cumple_normas_esperado,clinical_match
18,T19,[normas],p_depresion,0,0,0,0,False,False,True
19,T20,[normas],p_depresion,0,0,0,0,False,False,True
20,T21,[normas],p_depresion,0,0,0,100,True,True,True


In [21]:
#Revisión de fallos en casos emocionales
fallos = df2[df2["clinical_match"] == False]
fallos[summary_cols]

,id,categoria_esperada,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,cumple_normas_esperado,clinical_match
0,T01,[depresion],p_ansiedad,40,60,0,100,True,None,False
4,T05,[depresion],p_ansiedad,40,60,0,100,True,None,False
15,T16,[neutral],p_depresion,0,0,0,100,True,None,False
16,T17,[neutral],p_depresion,0,0,0,100,True,None,False
17,T18,[neutral],p_ansiedad,0,20,0,100,True,None,False


In [22]:
#Casos correctos
aciertos = df2[df2["clinical_match"] == True]
aciertos[summary_cols]

,id,categoria_esperada,predicted_top_clinical,p_depresion,p_ansiedad,p_suicida,score_normas,cumple_normas,cumple_normas_esperado,clinical_match
1,T02,[depresion],p_depresion,20,10,0,100,True,None,True
2,T03,"[depresion, ansiedad]",p_ansiedad,20,80,0,100,True,None,True
3,T04,"[suicida, depresion]",p_depresion,90,20,80,100,True,None,True
5,T06,[depresion],p_depresion,80,40,20,100,True,None,True
6,T07,"[depresion, ansiedad]",p_depresion,80,40,20,80,True,None,True
7,T08,"[depresion, suicida]",p_suicida,80,20,90,0,False,None,True
8,T09,"[ansiedad, depresion, suicida]",p_ansiedad,60,90,20,100,True,None,True
9,T10,[depresion],p_depresion,80,40,0,80,True,None,True
10,T11,"[depresion, suicida]",p_suicida,80,20,90,100,True,None,True
11,T12,"[depresion, suicida]",p_depresion,90,80,70,100,True,None,True


In [23]:
# Resumen de resultados
print("Total:", len(df2))
print("Aciertos clínicos:", int((df2["clinical_match"] == True).sum()))
print("Fallos clínicos:", int((df2["clinical_match"] == False).sum()))
print("No aplicables / normas sin comparación:", int(df2["clinical_match"].isna().sum()))

Total: 21
Aciertos clínicos: 16
Fallos clínicos: 5
No aplicables / normas sin comparación: 0
